# Step 3: Drivers’ analysis and identification: Criteria 2 and 3: Inequality assessment – Gini coefficient

Before running this notebook, please refer to the [Step 3 README](../Step_3/README.md) for instructions on the required input files and folder structure.

This notebook is organized into two main sections:
1. Loading the data files
2. Performing calculations

## Case study

- **Region:** Province of Quebec, Canada
- **Year:** 2021
- **Impact studied:** Climate change short-term
- **Available socioeconomic and demographic (SED) factors:** Income, Household type, Region and Tenure

## Required libraries

In [1]:
import os
import numpy as np
import pandas as pd

The following parameters define the study year, SED factor, household size of each subfactor and consumption categories. Modify these values for the case study

In [2]:
YEAR  = 2021 
FACTOR = "Income"
IMPACT = "CC"
HOUSEHOLDS_PER_FACTOR = ["Q1", "Q2", "Q3", "Q4", "Q5"]
HOUSEHOLD_SIZE = {
            "Q1": 1.22,
            "Q2": 1.67,
            "Q3": 2.04,
            "Q4": 2.93,
            "Q5": 3.51,
        }
HOUSEHOLD_SHARE = {
    "Q1": 0.2,
    "Q2": 0.2,
    "Q3": 0.2,
    "Q4": 0.2,
    "Q5": 0.2,
}
CATEGORIES = [
    "Clothing and accessories",
    "Food expenditures",
    "Health and personal care",
    "Household operations, furnishings and, equipment",
    "Miscellaneous expenditures",
    "Recreation",
    "Shelter",
    "Tobacco, alcohol and non-medical cannabis",
    "Transportation"]

## 1. Loading the data files

This notebook reads the environmental impact files generated in Step 2, one file per consumption category. The files are located in the {IMPACT}_by_category/ folder and are named using the first four letters of each category name.


In [3]:
folder = f"../Step_2/{IMPACT}_by_category"

data = {}
for category in CATEGORIES:
    short_name = category[:4]
    file_path = os.path.join(folder, f"{short_name}.xlsx")
    df_cat = pd.read_excel(file_path)
    data[category] = df_cat

## 2. Performing calculations

Criteria 2 and 3 assess the inequality of environmental impact distribution across SED subfactors using the Gini coefficient. Criterion 2 is calculated at the household level and Criterion 3 at the per capita level. A higher Gini coefficient indicates greater inequality in the distribution of impacts across subfactors.


Criterion 2 calculates the Gini coefficient of environmental impact distribution at the household level. The household share of each subfactor is specified in the parameters (HOUSEHOLD_SHARE).


In [4]:
# Criterion 2 - Gini at household level
gini_household_results = []

for category in CATEGORIES:
    df_cat = data[category]
    
    # Get impact values and household shares
    mt = df_cat.iloc[:, 1].values  # impact values
    segments = df_cat.iloc[:, 0].values  # subfactor labels
    
    D = np.array([HOUSEHOLD_SHARE[s] for s in segments])
    Y = mt / mt.sum()
    T = np.cumsum(Y)
    G = float((D * Y).sum() + 2 * (D * (1 - T)).sum() - 1)
    
    gini_household_results.append({
        "Category": category,
        "Gini (household)": round(G, 6)
    })

df_gini_household = pd.DataFrame(gini_household_results)
df_gini_household.to_excel(f"{YEAR}_{FACTOR}_Criterion2_Gini_household.xlsx", index=False)
df_gini_household

,Category,Gini (household)
0,Clothing and accessories,0.272234
1,Food expenditures,0.210376
2,Health and personal care,0.211407
3,"Household operations, furnishings and, equipment",0.215837
4,Miscellaneous expenditures,0.327708
5,Recreation,0.274481
6,Shelter,0.210556
7,"Tobacco, alcohol and non-medical cannabis",0.151347
8,Transportation,0.264831


Criterion 3 calculates the Gini coefficient at the per capita level. The population share of each subfactor is derived from the household share and the average household size.


In [5]:
# Criterion 3 - Gini at per capita level
gini_pc_results = []

for category in CATEGORIES:
    df_cat = data[category]
    
    segments = df_cat.iloc[:, 0].values
    mt = df_cat.iloc[:, 1].values
    
    # Calculate population shares
    pop = np.array([HOUSEHOLD_SHARE[s] * HOUSEHOLD_SIZE[s] for s in segments])
    D_pc = pop / pop.sum()
    
    # Per capita impact
    hh_sizes = np.array([HOUSEHOLD_SIZE[s] for s in segments])
    Y = mt / mt.sum()
    T = np.cumsum(Y)
    G = abs(float((D_pc * Y).sum() + 2 * (D_pc * (1 - T)).sum() - 1))
    
    gini_pc_results.append({
        "Category": category,
        "Gini (per capita)": round(G, 6)
    })

df_gini_pc = pd.DataFrame(gini_pc_results)
df_gini_pc.to_excel(f"{YEAR}_{FACTOR}_Criterion3_Gini_per_capita.xlsx", index=False)
df_gini_pc

,Category,Gini (per capita)
0,Clothing and accessories,0.065636
1,Food expenditures,0.009817
2,Health and personal care,0.011732
3,"Household operations, furnishings and, equipment",0.015626
4,Miscellaneous expenditures,0.133593
5,Recreation,0.074072
6,Shelter,0.008377
7,"Tobacco, alcohol and non-medical cannabis",0.061247
8,Transportation,0.063848
